In [13]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

# Create a SparkSession
spark = SparkSession.builder.appName("SparkSQL").getOrCreate()

def mapper(line):
    fields = line.split(',')
    return Row(ID=int(fields[0]), name=str(fields[1].encode("utf-8")), \
               age=int(fields[2]), numFriends=int(fields[3]))

lines = spark.sparkContext.textFile("fakefriends.csv")
people_rdd = lines.map(mapper)

# Infer the schema, and register the DataFrame as a table.
schemaPeople = spark.createDataFrame(people_rdd).cache()
schemaPeople.createOrReplaceTempView("people")

# SQL can be run over DataFrames that have been registered as a table. 
# spark.sql("select ...") returns a new DataFrame
teenagers = spark.sql("SELECT * FROM people WHERE age >= 13 AND age <= 19")
print("teenagers:")
teenagers.show(5)

# The results of SQL queries are RDDs and support all the normal RDD operations.
print("teenagers.collect():")
for teen in teenagers.collect():
  print(teen)

# We can also use functions instead of SQL queries:
print()
print("schemaPeople.groupBy('age').count().orderBy('age').show(5):")
schemaPeople.groupBy("age").count().orderBy("age").show(5)
spark.stop()


teenagers:
+---+----------+---+----------+
| ID|      name|age|numFriends|
+---+----------+---+----------+
| 21|  b'Miles'| 19|       268|
| 52|b'Beverly'| 19|       269|
| 54|  b'Brunt'| 19|         5|
|106|b'Beverly'| 18|       499|
|115|  b'Dukat'| 18|       397|
+---+----------+---+----------+
only showing top 5 rows

teenagers.collect():
Row(ID=21, name="b'Miles'", age=19, numFriends=268)
Row(ID=52, name="b'Beverly'", age=19, numFriends=269)
Row(ID=54, name="b'Brunt'", age=19, numFriends=5)
Row(ID=106, name="b'Beverly'", age=18, numFriends=499)
Row(ID=115, name="b'Dukat'", age=18, numFriends=397)
Row(ID=133, name="b'Quark'", age=19, numFriends=265)
Row(ID=136, name="b'Will'", age=19, numFriends=335)
Row(ID=225, name="b'Elim'", age=19, numFriends=106)
Row(ID=304, name="b'Will'", age=19, numFriends=404)
Row(ID=341, name="b'Data'", age=18, numFriends=326)
Row(ID=366, name="b'Keiko'", age=19, numFriends=119)
Row(ID=373, name="b'Quark'", age=19, numFriends=272)
Row(ID=377, name="b'Beve